# RAG Evaluation — HumanMaximizer Lead Gen

Evaluates faithfulness and answer relevance of the RAG pipeline using 5 demo leads.

**Prerequisites:**
- `docker compose up -d` with all services healthy
- `python scripts/ingest.py` completed (ChromaDB populated)
- `pip install ragas langchain-openai` (or use Ollama evaluation backend)

**Metrics evaluated:**
- `faithfulness` — does the generated answer contain only facts from the retrieved context?
- `answer_relevance` — is the generated answer relevant to the question asked?

In [ ]:
import sys
sys.path.insert(0, '../backend')

import os
os.environ.setdefault('DATABASE_URL', 'sqlite+aiosqlite:///./data/leads.db')

from rag.retriever import build_retriever, retrieve_chunks

retriever = build_retriever()
print('Retriever:', retriever)

In [ ]:
# 5 evaluation queries covering key HRMS selling points
EVAL_QUERIES = [
    "What payroll compliance features does HumanMaximizer offer for manufacturing companies?",
    "How does HumanMaximizer handle multi-location employee management?",
    "What statutory filing automation does HumanMaximizer provide?",
    "How does HumanMaximizer support onboarding for IT services companies?",
    "What reporting and analytics capabilities does HumanMaximizer HRMS include?",
]

retrieved = {}
for q in EVAL_QUERIES:
    chunks = retrieve_chunks(q, retriever) if retriever else []
    retrieved[q] = chunks
    print(f'Q: {q[:60]}...')
    print(f'  Retrieved {len(chunks)} chunks')
    if chunks:
        print(f'  First chunk: {chunks[0][:120]}...')
    print()

In [ ]:
# Generate answers using the local LLM via sales_node prompts
from langchain_ollama import OllamaLLM
from config import settings
from prompts import load_prompt

llm = OllamaLLM(
    model=settings.ollama_model,
    base_url=settings.ollama_base_url,
    temperature=0.1,
    num_ctx=2048,
)

answers = {}
for q in EVAL_QUERIES:
    context = '\n\n'.join(retrieved.get(q, []))
    if not context:
        answers[q] = 'No context retrieved.'
        continue
    prompt = (
        f'Using ONLY the following context, answer the question.\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {q}\n\n'
        f'Answer (2-3 sentences, no facts outside the context):'
    )
    answer = llm.invoke(prompt).strip()
    answers[q] = answer
    print(f'Q: {q[:60]}...')
    print(f'A: {answer[:200]}')
    print()

In [ ]:
# Build RAGAS dataset
from datasets import Dataset

data = {
    'question': list(answers.keys()),
    'answer': list(answers.values()),
    'contexts': [retrieved.get(q, []) for q in answers.keys()],
    'ground_truth': [
        'HumanMaximizer supports PF, ESI, TDS, and professional tax filing for manufacturing payroll.',
        'HumanMaximizer provides centralised HR management across multiple locations and plants.',
        'HumanMaximizer automates statutory filings including PF, ESI, and PT returns.',
        'HumanMaximizer provides automated offer-to-onboarding workflows for IT services companies.',
        'HumanMaximizer includes real-time headcount dashboards, attrition analytics, and custom reports.',
    ],
}

dataset = Dataset.from_dict(data)
print(f'Dataset created with {len(dataset)} rows')
dataset.to_pandas()

In [ ]:
# RAGAS evaluation
# NOTE: RAGAS by default uses OpenAI. For local-only evaluation,
# configure ragas to use OllamaLLM as the judge model.
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy

    results = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy],
    )
    print('RAGAS Results:')
    print(results)
    results.to_pandas()
except ImportError:
    print('ragas not installed. Run: pip install ragas')
    print('Showing raw answers instead:')
    import pandas as pd
    df = pd.DataFrame({
        'question': list(answers.keys()),
        'answer': list(answers.values()),
        'chunks_retrieved': [len(retrieved.get(q, [])) for q in answers.keys()],
    })
    display(df)